## Pydantic Graph


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(api_key=API_KEY, model="gemini-3.5-flash")
llm.invoke("salom nima yangiliklar").content[0]["text"]

In [ ]:
from langchain_core.messages import HumanMessage

llm.invoke([HumanMessage(content="salom nima yangiliklar")]).content[0]["text"]

In [ ]:
from pydantic import BaseModel
from typing import TypedDict, List


class graph_schema(BaseModel):
    user_message: str
    all_messages: List


In [ ]:
def generate_video_idea(schema: graph_schema):
    schema = schema.model_dump()
    user_message = schema["user_message"]
    response_ai = llm.invoke([HumanMessage(content=user_message)])
    schema["all_messages"] = [user_message] + [response_ai]
    return schema


def generate_video_prompt(schema: graph_schema):
    schema = schema.model_dump()
    user_message = schema["user_message"]
    response_ai = llm.invoke([HumanMessage(content=user_message)])
    schema["all_messages"] = [user_message] + [response_ai]
    return schema



In [ ]:
from langgraph.graph import START, END, StateGraph

graph = StateGraph(graph_schema)
graph.add_node("generate_video_idea", generate_video_idea)
graph.add_node("generate_video_prompt", generate_video_prompt)

graph.add_edge(START, "generate_video_idea")
graph.add_edge("generate_video_idea", "generate_video_prompt")
graph.add_edge("generate_video_prompt", END)

graph_pydantic = graph.compile()







In [ ]:
from IPython.display import Image, display

display(Image(graph_pydantic.get_graph().draw_mermaid_png()))

In [ ]:
natija = graph_pydantic.invoke({
    "user_message": "Salom yangi video yaratib ber",
    "all_messages": [],
})
natija